# 02 · M2 — Segmentación de instancias de vehículos (YOLO11-seg)
**IAM-811** · Laboratorio Final de Hito

Entrena un modelo propio de **5 clases**: `auto`, `bus`, `camion`, `minibus`, `moto`
(orden alfabético, el mismo con que Roboflow exporta). COCO no tiene *minibús*, por eso el notebook de la semana 3
propone un modelo propio: aquí se entrena con el dataset corregido en Roboflow (notebook 01).

**Contrato con `/inferencia`** (lo comprueba la sección 6 antes de exportar):
- archivo final **`vehiculos_seg.pt`** → `inferencia/pesos/vehiculos_seg.pt`
- `task = segment`; cada clase debe corresponder (por **nombre**, no por índice) a una de las 5 categorías.
  Se aceptan variantes como `Camión`, `Auto particular`, `van`, `trufi` (ver `inferencia/app/validacion.py`).

Mapeo a tarifas (tabla `tarifas` de la web):

| Clase | Tarifa base (Bs) | Ejes incluidos | Bs por eje adicional |
|---|---|---|---|
| auto | 5 | 2 | 0 |
| bus | 15 | 2 | 5 |
| camion | 20 | 2 | 10 |
| minibus | 8 | 2 | 0 |
| moto | 2 | 2 | 0 |

**GPU T4** · ~30–60 min para 100 épocas con ~500 imágenes.

In [ ]:
!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución > Cambiar tipo de entorno > GPU T4"
!pip -q install -U ultralytics==8.4.218 roboflow==1.7.3

## 1. Dataset
**Opción A — Roboflow (recomendada):** completa `WORKSPACE`, `PROYECTO` y `VERSION` de tu proyecto *Instance Segmentation*.
La API key se lee de los **Secretos de Colab** (icono de llave, nombre `ROBOFLOW_API_KEY`, con acceso al notebook activado);
nunca se escribe en el notebook.

**Opción B — zip:** sube `dataset_vehiculos.zip` generado en el notebook 01.

In [ ]:
import os, glob, yaml, shutil
FUENTE = 'roboflow'          # 'roboflow' o 'zip'
WORKSPACE, PROYECTO, VERSION = 'tu-workspace', 'peaje-vehiculos', 1
CLASES = ['auto', 'bus', 'camion', 'minibus', 'moto']
NOMBRE_FINAL = 'vehiculos_seg.pt'     # nombre EXACTO que busca inferencia/app/config.py

def preparar_yaml(ruta_yaml):
    """Deja el data.yaml con rutas absolutas validas en Colab (Roboflow exporta '../train/images', 'valid/', etc.)."""
    d = yaml.safe_load(open(ruta_yaml))
    base = os.path.dirname(os.path.abspath(ruta_yaml))
    d['path'] = base
    for clave, candidatas in (('train', ['train/images', 'images/train']),
                              ('val', ['valid/images', 'val/images', 'images/val']),
                              ('test', ['test/images', 'images/test'])):
        encontrada = next((c for c in candidatas if os.path.isdir(os.path.join(base, c))), None)
        if encontrada:
            d[clave] = encontrada
        elif clave == 'test':
            d.pop('test', None)
        else:
            raise FileNotFoundError(f'No hay carpeta {clave} en {base}. En Roboflow genera la version con split train/valid.')
    yaml.safe_dump(d, open(ruta_yaml, 'w'), allow_unicode=True, sort_keys=False)
    return d

if FUENTE == 'roboflow':
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    ds = rf.workspace(WORKSPACE).project(PROYECTO).version(VERSION).download('yolov11', location='/content/ds_vehiculos')
    DATA_YAML = f'{ds.location}/data.yaml'
else:
    from google.colab import files
    for n in files.upload():
        !unzip -qo "$n" -d /content
    DATA_YAML = '/content/dataset_vehiculos/data.yaml'

datos = preparar_yaml(DATA_YAML)
print('Clases del dataset:', datos['names'])
print('train:', datos['train'], '| val:', datos['val'])

## 2. Normalizar y verificar nombres de clase
Roboflow a veces exporta "Auto particular", "Camión", "van", etc. Se renombran **sin cambiar los índices** (las etiquetas
`.txt` siguen siendo válidas). Si alguna clase no corresponde a las 5 categorías, la celda se detiene con un error claro.

In [ ]:
NORMALIZAR = {'car': 'auto', 'auto particular': 'auto', 'automovil': 'auto', 'automóvil': 'auto', 'vagoneta': 'auto',
              'camioneta': 'auto', 'sedan': 'auto', 'suv': 'auto', 'pickup': 'auto',
              'autobus': 'bus', 'autobús': 'bus', 'flota': 'bus',
              'camión': 'camion', 'truck': 'camion', 'trailer': 'camion',
              'minibús': 'minibus', 'van': 'minibus', 'micro': 'minibus', 'microbus': 'minibus', 'trufi': 'minibus',
              'motorcycle': 'moto', 'motocicleta': 'moto', 'motorbike': 'moto'}

def normalizar(n):
    n = str(n).strip().lower().replace('_', ' ').replace('-', ' ')
    return NORMALIZAR.get(n, n)

nombres = datos['names'] if isinstance(datos['names'], list) else [datos['names'][k] for k in sorted(datos['names'])]
datos['names'] = [normalizar(n) for n in nombres]
desconocidas = [n for n in datos['names'] if n not in CLASES]
assert not desconocidas, f'Clases que no son del peaje: {desconocidas}. Renómbralas/elimínalas en Roboflow.'
faltan = set(CLASES) - set(datos['names'])
print('Clases normalizadas:', datos['names'], '| faltan:', faltan or 'ninguna')
if faltan:
    print('ADVERTENCIA: el modelo no aprenderá', faltan, '- agrega ejemplos de esas clases si puedes.')
datos['nc'] = len(datos['names'])
yaml.safe_dump(datos, open(DATA_YAML, 'w'), allow_unicode=True, sort_keys=False)

## 3. Entrenamiento
Se parte de `yolo11s-seg.pt` (COCO) por transferencia de aprendizaje. `imgsz=640` debe coincidir con
`IMGSZ_PROPIO=640` del servidor de inferencia. Con más GPU puedes usar `yolo11m-seg.pt`.

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolo11s-seg.pt')
res = modelo.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16, patience=25, device=0,
                   fliplr=0.5, hsv_v=0.3, degrees=3, mosaic=1.0, close_mosaic=10,
                   project='/content/runs/seg', name='vehiculos', exist_ok=True)
BEST = '/content/runs/seg/vehiculos/weights/best.pt'

## 4. Evaluación (mAP de cajas y máscaras, por clase)

In [ ]:
import pandas as pd
from IPython.display import Image, display
m = YOLO(BEST).val(data=DATA_YAML, split='val', plots=True)
print(f"mAP50 caja: {m.box.map50:.3f} | mAP50-95 caja: {m.box.map:.3f}")
print(f"mAP50 máscara: {m.seg.map50:.3f} | mAP50-95 máscara: {m.seg.map:.3f}")
display(pd.DataFrame([{'clase': m.names[i], 'mAP50-95 máscara': round(float(m.seg.maps[i]), 3)} for i in range(len(m.names))]))
for f in ('confusion_matrix_normalized.png', 'MaskPR_curve.png', 'results.png'):
    ruta = sorted(glob.glob(f'/content/runs/**/{f}', recursive=True), key=os.path.getmtime)
    if ruta:
        display(Image(ruta[-1], width=700))

## 5. Prueba sobre una imagen

In [ ]:
import matplotlib.pyplot as plt
!wget -q -O prueba.jpg https://ultralytics.com/images/bus.jpg
r = YOLO(BEST).predict('prueba.jpg', conf=0.3, imgsz=640, verbose=False)[0]
plt.figure(figsize=(10, 7)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis('off'); plt.show()

## 6. Verificar contrato y exportar `vehiculos_seg.pt`
Comprueba lo mismo que el servidor al arrancar (`inferencia/app/validacion.py`). Luego descarga el archivo
y, si quieres, lo copia a Google Drive en `MyDrive/peaje/pesos/` (la carpeta que lee `colab_servidor.ipynb`).

In [ ]:
final = YOLO(BEST)
nombres_finales = [final.names[k] for k in sorted(final.names)]
assert final.task == 'segment', f'task={final.task}: debe ser segment'
assert all(normalizar(n) in CLASES for n in nombres_finales), f'Clases no válidas: {nombres_finales}'
print('OK contrato M2 · task=segment · clases =', nombres_finales)

shutil.copy(BEST, f'/content/{NOMBRE_FINAL}')
GUARDAR_DRIVE = True
if GUARDAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/peaje/pesos', exist_ok=True)
    shutil.copy(f'/content/{NOMBRE_FINAL}', f'/content/drive/MyDrive/peaje/pesos/{NOMBRE_FINAL}')
    print('Guardado en Drive: MyDrive/peaje/pesos/' + NOMBRE_FINAL)
from google.colab import files
files.download(f'/content/{NOMBRE_FINAL}')